# CDC-1 — Local Postgres & logical replication

**Break → Detect → Fix → Prove.** Change Data Capture doesn't poll your tables — it reads the
database's **write-ahead log (WAL)**. Postgres exposes row-level changes from the WAL via **logical
decoding**, gated by two objects: a **publication** (which tables) and a **replication slot** (a
named cursor into the WAL that the consumer advances). This module stands those up by hand so the
moving parts are visible before Debezium automates them in [CDC-2](../connector_bringup/).

**Prerequisite:** `make cdc-up` (starts Postgres + Kafka Connect; see the [track README](../README.md)).
**Laptop-safe:** a 20-row table, all dropped at teardown.

In [ ]:
from common import cdc_helpers as cdc

print("Postgres :", f"{cdc.PG_HOST}:{cdc.PG_PORT}", "db", cdc.PG_DB)
# wal_level must be 'logical' for row-level decoding (set via docker-compose command flags).
print("wal_level           :", cdc.pg_exec("SHOW wal_level", fetch=True)[0][0])
print("max_replication_slots:", cdc.pg_exec("SHOW max_replication_slots", fetch=True)[0][0])
print("max_wal_senders     :", cdc.pg_exec("SHOW max_wal_senders", fetch=True)[0][0])

## 1. A source table + seed data

Our "production" OLTP table. Debezium will later mirror it into Iceberg; for now it's just a
normal Postgres table.

In [ ]:
cdc.pg_exec("DROP TABLE IF EXISTS public.cdc1_orders")
cdc.seed_orders("cdc1_orders", n=20)
print("seeded:", cdc.pg_exec("SELECT count(*), min(id), max(id) FROM public.cdc1_orders", fetch=True)[0])
print("sample:", cdc.pg_exec("SELECT id, customer, amount, status FROM public.cdc1_orders ORDER BY id LIMIT 3", fetch=True))

## 2. The two logical-replication objects

Debezium creates these for you, but doing it by hand shows what they *are*:

- **Publication** — the set of tables whose changes are published (`CREATE PUBLICATION`).
- **Replication slot** — a named position in the WAL. Postgres **retains every WAL segment** from
  the slot's `restart_lsn` forward until a consumer reads past it. No consumer ⇒ WAL piles up
  (the CDC-5 pathology). The `pgoutput` plugin is built into Postgres 16.

In [ ]:
cdc.pg_exec("DROP PUBLICATION IF EXISTS cdc1_pub")
cdc.pg_exec("CREATE PUBLICATION cdc1_pub FOR TABLE public.cdc1_orders")
cdc.drop_slot("cdc1_slot")  # in case a previous run left it
cdc.pg_exec("SELECT pg_create_logical_replication_slot('cdc1_slot', 'pgoutput')", fetch=True)

print("publications:", [r[0] for r in cdc.pg_exec("SELECT pubname FROM pg_publication", fetch=True)])
for s in cdc.list_slots():
    print("slot:", s)

## 3. Detect — read the slot's state

`pg_replication_slots` is the source of truth. `active=False` means **no consumer is attached**;
`retained_bytes` is how much WAL this slot is currently pinning.

In [ ]:
before = cdc.list_slots()[0]
print("slot before writes:", before)

# Write more rows: the WAL advances, but with no consumer the slot's restart_lsn does NOT,
# so the WAL it must retain grows.
for i in range(20, 60):
    cdc.pg_exec("INSERT INTO public.cdc1_orders(id,customer,amount,status) VALUES (%s,%s,%s,'NEW')",
                (i, f"cust-{i%7}", 10.0+i))
after = cdc.list_slots()[0]
print("slot after  writes:", after)
print(f"retained WAL grew: {before['retained_bytes']} -> {after['retained_bytes']} bytes (slot is inactive)")

## 4. Diagnose & Prove

The slot is **inactive** yet its `retained_bytes` climbs as we write — Postgres can't recycle that
WAL because, as far as it knows, a consumer still needs it. That's the whole basis of CDC:

- a **healthy** consumer (Debezium) advances the slot, WAL is recycled, disk stays flat;
- a **stalled** consumer (or an orphaned slot) pins WAL forever → disk fills. We deliberately
  trigger and detect that in **CDC-5** using exactly this `list_slots()` metric.

## 5. Takeaways & "in real production…"

- **`wal_level=logical`** is the price of CDC; set it before you need it (it requires a restart).
- A replication slot is **durable state on the primary** — an abandoned slot is an outage waiting
  to happen. Always monitor `pg_replication_slots` (`active`, `retained_bytes`) and alert on growth.
- One slot per consumer; drop slots you no longer use. `max_slot_wal_keep_size` (PG 13+) caps the
  damage by letting Postgres drop a slot rather than run out of disk.

## Teardown

In [ ]:
cdc.drop_slot("cdc1_slot")
cdc.pg_exec("DROP PUBLICATION IF EXISTS cdc1_pub")
cdc.pg_exec("DROP TABLE IF EXISTS public.cdc1_orders")
print("slots now:", cdc.list_slots(), "| make clean clears generated data")